# CAN Bus Message Testing Tool

This notebook demonstrates:

1. CAN message simulation
2. ECU data Validation
3. Message integrity testing
4. DBC based CAN decoding using cantools
5. CAN log replay testing
6. Automated ECU fault injection testing

In [ ]:
!pip install python-can cantools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.6/159.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.1.2
    Uninstalling wrapt-2.1.2:
      Successfully uninstalled wrapt-2.1.2


### 1. CAN Message Simulation
This module simulates CAN bus communication using the `python-can` library in a virtual environment.
It enables testing of ECU message transmission without requiring physical CAN hardware.
The simulation mimics real-world arbitration IDs and payload structures used in automotive systems.
This helps validate message flow, timing behavior, and integration logic early in development.

In [ ]:
import can

# Use updated parameter + same channel
bus = can.Bus(interface='virtual', channel='test_channel')

msg = can.Message(
    arbitration_id=0x123,
    data=[10, 20, 30, 40],
    is_extended_id=False
)

bus.send(msg)
print("Message Sent:", msg)

# Receive from same channel
received = bus.recv(timeout=1)
print("Message Received:", received)

# Proper shutdown
bus.shutdown()

Message Sent: Timestamp:        0.000000    ID:      123    S Rx                DL:  4    0a 14 1e 28
Message Received: None


### 2. ECU Data Validation
This feature validates ECU signal data against predefined constraints and safety thresholds.
It ensures that parameters such as speed, RPM, or temperature remain within acceptable limits.
Assertions and validation logic help detect anomalies and prevent unsafe conditions.
This is critical for functional safety and compliance with automotive standards.

In [ ]:
def validate_speed(speed):
    assert 0 <= speed <= 120, "Speed out of range"
    return "Valid Speed"

ecu_data = {"speed": 85}

result = validate_speed(ecu_data["speed"])
print(result)

Valid Speed


### 3. Message Integrity Testing
This module verifies the integrity of CAN messages using checksum-based validation techniques.
It ensures that transmitted data is not corrupted during communication across the network.
Integrity checks help identify transmission errors and maintain system reliability.
Such validation is essential in safety-critical automotive applications.

In [ ]:
def checksum(data):
    return sum(data) % 256

data = [10, 20, 30]
cs = checksum(data)

print("Checksum:", cs)

if cs == checksum(data):
    print("Integrity Verified")

Checksum: 60
Integrity Verified


### 4. DBC-Based CAN Decoding
This feature decodes raw CAN messages into human-readable signals using a DBC database via `cantools`.
DBC files define message structure, signal scaling, and encoding rules used by ECUs.
Decoding allows engineers to interpret vehicle data such as speed, torque, and sensor values.
It plays a key role in diagnostics, validation, and reverse engineering workflows.

In [ ]:
import cantools

db = cantools.database.load_string("""
BO_ 123 SpeedMsg: 8 Vector__XXX
 SG_ Speed : 0|8@1+ (1,0) [0|255] "" Vector__XXX
""")

msg = db.get_message_by_name("SpeedMsg")
decoded = msg.decode(bytes([100,0,0,0,0,0,0,0]))

print("Decoded:", decoded)

Decoded: {'Speed': 100}


### 5. CAN Log Replay Testing
This module replays recorded CAN traffic logs to simulate real-world driving scenarios.
It is useful for regression testing and validating ECU behavior against historical data.
Replay testing helps reproduce edge cases and intermittent issues reliably.
This approach improves test coverage without requiring live vehicle testing.

In [ ]:
log = [
    {"id": 0x101, "data": [1,2]},
    {"id": 0x102, "data": [3,4]}
]

for frame in log:
    print("Replaying:", frame)

Replaying: {'id': 257, 'data': [1, 2]}
Replaying: {'id': 258, 'data': [3, 4]}


### 6. Automated ECU Fault Injection Testing
This feature introduces controlled faults into CAN messages to test ECU robustness.
Faults may include corrupted data, extreme values, or unexpected signal behavior.
It helps evaluate how the system responds to abnormal or failure conditions.
This is essential for validating fault tolerance and safety mechanisms in automotive systems.

In [ ]:
def inject_fault(data):
    data[0] = 255  # fault injection
    return data

normal = [10, 20, 30]
faulty = inject_fault(normal.copy())

print("Normal:", normal)
print("Faulty:", faulty)

Normal: [10, 20, 30]
Faulty: [255, 20, 30]


### Conclusion
This project demonstrates a comprehensive CAN bus testing framework covering simulation, validation, decoding, and fault injection.
It provides a scalable approach to verify ECU communication and ensure data integrity in automotive systems.
The use of virtual CAN enables efficient development and testing without physical hardware dependencies.
Integration with DBC decoding and replay testing enhances real-world applicability and test coverage.
This framework can be extended for advanced ADAS validation and integrated into CI/CD pipelines for automation.